# J-lens replication on Kaggle T4×2

Companion to [anthropics/jacobian-lens](https://github.com/anthropics/jacobian-lens) and the paper *Verbalizable Representations Form a Global Workspace in Language Models* (Anthropic, 2026-07-06).

**Runtime:** Kaggle → Settings → Accelerator → **GPU T4×2**, Internet **On**. Add your `HF_TOKEN` as a Kaggle secret if you plan to use gated models (Llama, some Gemma variants).

**Cost:** free. Fitting a 4B model at n=25 prompts takes ~30 min. You have ~30 GPU hours/week on Kaggle.

## 1. Install

In [ ]:
!git clone -q https://github.com/anthropics/jacobian-lens.git
!pip -q install -e ./jacobian-lens datasets safetensors
!pip -q install -U 'transformers>=5.5'

In [ ]:
import os
# If using gated models, uncomment and set via Kaggle secrets
# from kaggle_secrets import UserSecretsClient
# os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

import torch, transformers, jlens
print('CUDA:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}]', torch.cuda.get_device_name(i))

## 2. Config — pick your target model

Start with Qwen 3.6-4B (Apache 2.0, no gate, fits on a single T4). Swap `MODEL` for any HF decoder — `jlens.hf._LAYOUTS` already covers Llama, Qwen, Mistral, Gemma, OLMo, StableLM, Phi, GPT-2, and Pythia.

In [ ]:
MODEL = 'Qwen/Qwen3.6-4B'
N_PROMPTS = 25
MAX_SEQ_LEN = 128
SKIP_FIRST = 4
TARGET_LAYER = -2      # penultimate block, paper-faithful
DIM_BATCH = 4          # 4 on T4 (16 GB), 8 on L4 (24 GB), 16 on A100+
DTYPE = torch.bfloat16
ATTN = 'sdpa'          # NEVER 'flash_attention_2' — breaks batched autograd
OUT_LENS = f'/kaggle/working/{MODEL.replace("/", "_")}-jlens.pt'

## 3. Load model + wrap

In [ ]:
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=DTYPE, attn_implementation=ATTN,
    low_cpu_mem_usage=True, token=os.environ.get('HF_TOKEN')
).to('cuda')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL, token=os.environ.get('HF_TOKEN'))

model = jlens.from_hf(hf_model, tokenizer, compile=True)
print(f'n_layers={model.n_layers}  d_model={model.d_model}')

## 4. Background corpus (Neel Nanda's pile-10k subset)

In [ ]:
from datasets import load_dataset
ds = load_dataset('NeelNanda/pile-10k', split='train')
prompts = []
for row in ds:
    if isinstance(row['text'], str) and len(row['text']) > 200:
        prompts.append(row['text'])
        if len(prompts) >= N_PROMPTS:
            break
print(f'Selected {len(prompts)} prompts')

## 5. Fit the J-lens

Watch VRAM. If you OOM, drop `DIM_BATCH` to 2. Progress prints per prompt.

In [ ]:
import time
t0 = time.time()
lens = jlens.fit(
    model, prompts=prompts,
    target_layer=TARGET_LAYER, dim_batch=DIM_BATCH,
    max_seq_len=MAX_SEQ_LEN, skip_first=SKIP_FIRST,
    checkpoint_path=OUT_LENS.replace('.pt', '.ckpt.pt'),
)
print(f'Fit done in {(time.time()-t0)/60:.1f} min')
print(lens)
lens.save(OUT_LENS)
print('Peak VRAM:', torch.cuda.max_memory_allocated()/1e9, 'GB')

## 6. Sanity-check readout

Should surface the intermediate concept ("spider", "Paris", …) at mid layers even though it never appears in the prompt.

In [ ]:
probes = [
    'The number of legs on the animal that spins webs is',
    'The capital of the country shaped like a boot is',
    'Fact: The currency used in the country whose flag has a red maple leaf is the',
]
for prompt in probes:
    lens_logits, _, _ = lens.apply(model, prompt, positions=[-2])
    print('\nprompt:', prompt)
    for layer in sorted(lens_logits)[::max(1, len(lens_logits)//8)]:
        top = lens_logits[layer][0].topk(5).indices.tolist()
        print(f'  L{layer:>2}:', [tokenizer.decode([t]) for t in top])

## 7. Run the six lens-quality evals

In [ ]:
import json, pathlib
evals_dir = pathlib.Path('jacobian-lens/data/evaluations')
for path in sorted(evals_dir.glob('lens-eval-*.json')):
    data = json.load(path.open())
    print(path.name, '→', len(data['items']), 'items')

In [ ]:
# Then use the scripts/run_evals.py logic (in the companion repo) to score pass@k.
# For a self-contained quick check, score multihop only:

def tokens_of(word):
    ids = []
    for v in {word, ' '+word, word.lower(), ' '+word.lower()}:
        e = tokenizer(v, add_special_tokens=False).input_ids
        if len(e) == 1: ids.append(e[0])
    return sorted(set(ids))

def score_eval(path, k=(1,5,10)):
    items = json.load(path.open())['items']
    hits = {ki: [] for ki in k}
    for item in items:
        lens_logits, _, _ = lens.apply(model, item['prompt'], positions=[-1])
        inters = item['intermediates'] if isinstance(item['intermediates'], list) else [item['intermediates']]
        item_hits = {ki: 0 for ki in k}
        total = 0
        for inter in inters:
            tok_ids = tokens_of(inter if isinstance(inter, str) else inter[0])
            if not tok_ids: continue
            best_rank = min(
                int(logits[0].argsort(descending=True).tolist().index(t))+1
                for _, logits in lens_logits.items() for t in tok_ids
            )
            for ki in k:
                if best_rank <= ki: item_hits[ki] += 1
            total += 1
        if total:
            for ki in k: hits[ki].append(item_hits[ki]/total)
    return {f'pass@{ki}': sum(hits[ki])/max(1,len(hits[ki])) for ki in k}

print('multihop:', score_eval(evals_dir / 'lens-eval-multihop.json'))

## 8. Publish to the Hub

Once you have a lens you trust, push it so others (and your ZeroGPU Space) can load it via `JacobianLens.from_pretrained(<repo_id>)`.

In [ ]:
from huggingface_hub import HfApi, create_repo
REPO = f'<your-hf-username>/{MODEL.split("/")[-1].lower()}-jlens'
create_repo(REPO, exist_ok=True, token=os.environ['HF_TOKEN'])
HfApi(token=os.environ['HF_TOKEN']).upload_file(
    path_or_fileobj=OUT_LENS, path_in_repo='lens.pt', repo_id=REPO
)
print('Pushed →', f'https://huggingface.co/{REPO}')